# 04 · LLM prompting

Asks a small instruct LLM, Qwen2.5-1.5B-Instruct, to label each test post: first with the instructions alone (**zero-shot**), then with six labeled example posts added (**few-shot**). Both are scored with `src/evaluate.py`, next to the baselines from notebook 03. Stage 5 fine-tunes this same model, so these two runs show what it can do before any training.

- **Inputs:** `data/test_hand_labeled.csv` (from notebook 02) and the prompt in `src/prompts.py`
- **Outputs:** `results/preds_llm_zeroshot.csv` and `preds_llm_fewshot.csv`, plus the model's raw replies in `replies_llm_zeroshot.csv` and `replies_llm_fewshot.csv`

The model downloads from Hugging Face the first time (about 3 GB). It runs on the Mac's GPU and takes about 6 minutes for both prompts.

## Setup

In [1]:
import logging
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

# Hide two harmless notices printed when transformers is imported: one about notebook
# progress bars, and one from torch saying a feature we don't use isn't supported on macOS
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(lambda record: False)
from huggingface_hub.utils import logging as hub_logging
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging

# Let the notebook import our shared code from src/
ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src.evaluate import RESULTS_DIR, evaluate, load_test, save_preds, score_table
from src.labels import LABELS, parse_reply
from src.prompts import FEW_SHOT_EXAMPLES, build_messages

# Hide Hugging Face notices about login tokens and download progress; the model is public
hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"  # MPS is the Apple Silicon GPU
MODELS = ["author_tag", "vader", "finbert", "cryptobert", "llm_zeroshot", "llm_fewshot"]

test = load_test()

## 1. Load the model

An LLM writes text one **token** (a word or piece of a word) at a time, each time predicting the most likely next token. A *base* model only continues text. An **instruct model** has been trained further on conversations of instructions and answers, so it follows requests like *"answer with one word"*.

**Qwen2.5-1.5B-Instruct** is a 1.5-billion-parameter instruct model from Alibaba. It was chosen because its weights are open, it needs no access request, and it's small enough to fine-tune on a free Colab GPU in Stage 5. In a short check on validation posts, it answered with a valid label every time, so the larger 3B version isn't needed.

The weights are loaded as 16-bit numbers (`bfloat16`), which takes half the memory of the usual 32-bit, about 3 GB.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(DEVICE)
model.eval()  # inference only, no training
print(f"{model.num_parameters() / 1e9:.2f}B parameters on {DEVICE}")

1.54B parameters on mps


## 2. The prompt

The prompt lives in `src/prompts.py`, because the fine-tune (Stage 5) and local inference (Stage 7) must use exactly the same words. Its label definitions come from the hand-labeling guide in notebook 02, so the model is asked the same question the test set answers.

- **Zero-shot:** the model gets the instructions and the post, with no examples.
- **Few-shot:** the same, but first the model sees six example posts with their correct labels, 2 per class, written as earlier turns of the conversation. The same six examples are used for every post. They come from `train.csv`, so none is in the test set.

A chat model expects a conversation, not plain text. The tokenizer's **chat template** turns the messages (a *system* message setting the task, then the *user* message) into one string with the special markers the model was trained on. This is exactly what the model sees for one post in zero-shot:

In [3]:
messages = build_messages("loaded more, see you at 50k")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<|im_start|>system
You classify the sentiment of crypto social media posts.<|im_end|>
<|im_start|>user
Classify this crypto post as bullish, bearish or neutral.

- bullish: the author expects or wants the price to rise, is buying or holding, or celebrates gains.
- bearish: the author expects the price to fall, is selling or shorting, warns others, or calls a coin a scam.
- neutral: no clear lean, such as news, facts, plain questions, greetings, ads or spam.

Post: loaded more, see you at 50k

Answer with one word: bullish, bearish or neutral.<|im_end|>
<|im_start|>assistant



The prompt ends with `<|im_start|>assistant`, so the model's next tokens are its answer.

The six few-shot examples, in the order the model sees them. Each class has one post with slang and one without. #1 and #3 show that a slang word alone doesn't decide the label: *"fud"* in a bullish post and *"hodl"* in a neutral headline.

In [4]:
pd.set_option("display.max_colwidth", None)  # show whole posts in tables
pd.DataFrame(FEW_SHOT_EXAMPLES, index=range(1, 7))[["label", "text"]]

,label,text
1,bullish,"fud will come and go, doge is here to stay."
2,bearish,all faith is gone. crypto to be worthless.
3,neutral,new article : ethereum classic – this may be the answer to the question ‘to hodl or sell'
4,bearish,biggest rug pull in history of financial markets incoming 🤘🏼
5,neutral,has anyone successfully sent cro from coin base to the cro app?
6,bullish,first green candle all day…keep them coming


## 3. Getting a label from the model

**Greedy decoding.** At each step, the model gives every possible next token a probability. By default, Qwen *samples* from those probabilities (temperature 0.7), so the same post can get a different answer each run. **Greedy decoding** always takes the single most likely token instead, so every run gives the same replies and the results can be reproduced.

Qwen's defaults also include a *repetition penalty*, which makes tokens already in the prompt less likely. Here it would push the model away from the very label words it is asked to use, and in few-shot those words appear six times in the examples. So it is turned off (`1.0`).

The answer should be one word, but up to 10 new tokens are allowed, so a reply that doesn't follow the format can still be read later.

In [5]:
settings = model.generation_config
settings.do_sample = False  # greedy decoding
settings.temperature = None  # temperature, top_p and top_k only matter when sampling
settings.top_p = None
settings.top_k = None
settings.repetition_penalty = 1.0  # off
settings.max_new_tokens = 10

In [6]:
def classify(text, few_shot=False):
    """Ask the LLM to label one post. Returns its raw reply."""
    messages = build_messages(text, few_shot)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    output = model.generate(**inputs)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]  # generate returns the prompt + the reply
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

**The reply parser.** `parse_reply` in `src/labels.py` turns a reply into a label. It ignores capitals and punctuation, and accepts the reply only if exactly one of the three label words appears in it. Anything else, such as *"positive"* or *"bullish or bearish"*, becomes `INVALID`, which the scorer counts as wrong.

A quick test on the three examples from the labeling guide:

In [7]:
for text in ["loaded more, see you at 50k", "this is heading to 20k, get out while you can", "what time does the fed speak today?"]:
    reply = classify(text)
    print(f"{reply!r:12} -> {parse_reply(reply):8} {text}")

'bullish'    -> bullish  loaded more, see you at 50k


'Bullish'    -> bullish  this is heading to 20k, get out while you can


'neutral'    -> neutral  what time does the fed speak today?


It already gets one wrong: *"this is heading to 20k, get out while you can"* is bearish, but the model reads it as bullish, perhaps taking *"20k"* as a price target. It also capitalizes some answers and not others, which the parser handles.

## 4. Zero-shot

The model labels each of the 300 test posts, one at a time. The raw replies are saved for error analysis, and the parsed labels go to the usual prediction file.

In [8]:
def run_prompt(few_shot):
    """Ask the LLM about every test post. Returns the raw replies."""
    start = time.time()
    replies = [classify(text, few_shot) for text in test["text"]]
    seconds = time.time() - start
    print(f"{len(replies)} posts in {seconds / 60:.1f} minutes ({seconds / len(replies):.2f} s per post)")
    return replies


def save_replies(replies, model_name):
    """Save the raw replies (for error analysis) and the parsed labels (for scoring)."""
    preds = [parse_reply(reply) for reply in replies]
    pd.DataFrame({"post_id": test["post_id"], "reply": replies, "pred_label": preds}).to_csv(
        RESULTS_DIR / f"replies_{model_name}.csv", index=False
    )
    save_preds(test["post_id"], preds, model_name)

In [9]:
zeroshot_replies = run_prompt(few_shot=False)
save_replies(zeroshot_replies, "llm_zeroshot")
pd.Series(zeroshot_replies).value_counts().head(10)  # the most common raw replies

300 posts in 1.4 minutes (0.28 s per post)


bearish    110
Bullish    103
bullish     51
neutral     36
Name: count, dtype: int64

## 5. Few-shot

The same posts, with the six examples added to every prompt. The prompt is about six times longer (roughly 870 tokens instead of 135), so each post takes longer.

In [10]:
fewshot_replies = run_prompt(few_shot=True)
save_replies(fewshot_replies, "llm_fewshot")
pd.Series(fewshot_replies).value_counts().head(10)

300 posts in 4.7 minutes (0.95 s per post)


bearish    159
neutral     89
bullish     52
Name: count, dtype: int64

## 6. Results

Macro-F1 for every model so far, on all posts and on each group. CryptoBERT's score is likely optimistic, since it was trained on the same StockTwits data (see notebook 03).

In [11]:
table = score_table(MODELS)
table.pivot(index="model", columns="group", values="macro_f1").loc[MODELS, ["all", "slang-heavy", "plain"]].round(3)

group,all,slang-heavy,plain
model,,,
author_tag,0.453,0.502,0.403
vader,0.396,0.402,0.382
finbert,0.366,0.298,0.421
cryptobert,0.466,0.475,0.441
llm_zeroshot,0.517,0.501,0.521
llm_fewshot,0.538,0.536,0.534


Invalid-output rate for the two LLM runs (the other models always return a label):

In [12]:
table[table["model"].str.startswith("llm")].pivot(index="model", columns="group", values="invalid_rate")[["all", "slang-heavy", "plain"]]

group,all,slang-heavy,plain
model,,,
llm_fewshot,0.0,0.0,0.0
llm_zeroshot,0.0,0.0,0.0


F1 for each label, on all posts:

In [13]:
pd.DataFrame({model: evaluate(model)["all"]["per_class"]["f1"] for model in MODELS}).T.loc[MODELS, LABELS].round(3)

label,bullish,bearish,neutral
author_tag,0.458,0.469,0.432
vader,0.368,0.395,0.426
finbert,0.136,0.294,0.668
cryptobert,0.559,0.480,0.358
llm_zeroshot,0.644,0.558,0.349
llm_fewshot,0.534,0.507,0.573


Confusion matrices on all posts (rows are the hand labels, columns the predictions):

In [14]:
for model_name in ["llm_zeroshot", "llm_fewshot"]:
    print(model_name)
    display(evaluate(model_name)["all"]["confusion"])

llm_zeroshot


predicted,bullish,bearish,neutral,INVALID
true,,,,
bullish,75,4,0,0
bearish,12,48,2,0
neutral,67,58,34,0


llm_fewshot


predicted,bullish,bearish,neutral,INVALID
true,,,,
bullish,35,31,13,0
bearish,1,56,5,0
neutral,16,72,71,0


Few-shot changed many answers. The table counts posts by zero-shot answer (rows) and few-shot answer (columns):

In [15]:
zeroshot = [parse_reply(reply) for reply in zeroshot_replies]
fewshot = [parse_reply(reply) for reply in fewshot_replies]
pd.crosstab(pd.Series(zeroshot, name="zero-shot"), pd.Series(fewshot, name="few-shot"))

few-shot,bearish,bullish,neutral
zero-shot,,,
bearish,102,0,8
bullish,57,52,45
neutral,0,0,36


## Summary

- **Both prompts beat every baseline without any training.** Macro-F1 is 0.52 for zero-shot and 0.54 for few-shot, against 0.47 for CryptoBERT, the best baseline. The 1.5B model answered with a valid label for all 600 prompts.
- **There's no slang gap.** Both prompts score about the same on slang-heavy and plain posts, unlike FinBERT (0.30 vs. 0.42).
- **Zero-shot rarely answers neutral.** It finds 75 of the 79 bullish posts and 48 of the 62 bearish ones, but calls only 34 of the 159 neutral posts neutral. Like the author tags and CryptoBERT, it reads a lean into posts that have none.
- **Few-shot finds more neutral posts but leans bearish.** It gets 71 neutral posts right, but answers bearish for 159 of the 300 posts. That includes 31 bullish posts, some of them clear, like *"cup and handle forming on the live this thing bout to explode!"*. Small models are known to react strongly to which examples they are shown. The six examples stay fixed anyway: picking them by test score would be tuning on the test set.
- **Few-shot mostly trades one kind of mistake for another.** The two prompts agree on only 63% of posts, yet few-shot gets only 5 more posts right (162 vs. 157), and a 0.02 gap in macro-F1 is small for 300 posts. Stage 5 fine-tunes this model on 30k posts with the zero-shot prompt. One thing to watch: the training labels are author tags, which also over-read sentiment, so the fine-tune may find even fewer neutral posts.